In [16]:

# ============================================================
# NOTEBOOK 01
# RINEX DATA INGESTION AND PARSER DEVELOPMENT
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

# ------------------------------------------------------------
# 1. PROJECT DIRECTORY
# ------------------------------------------------------------

PROJECT_DIR = Path(
    r"C:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final"
)

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
OUTPUT_DIR = PROJECT_DIR / "Output" / "Notebook_01"

# ------------------------------------------------------------
# 2. INPUT DATASET
# ------------------------------------------------------------

MAIN_STATION = "BAKO"
MAIN_DOY = 300

MAIN_RINEX_FILE = (
    RAW_DATA_DIR
    / f"DOY{MAIN_DOY}"
    / f"bako{MAIN_DOY}0.03o"
)

# ------------------------------------------------------------
# 3. OUTPUT DIRECTORY STRUCTURE
# ------------------------------------------------------------

OUTPUT_SUBDIRS = [
    "header_metadata",
    "epoch_structure",
    "parsed_observations",
    "diagnostics",
    "parsing_logs"
]

# ------------------------------------------------------------
# 4. BASIC DIRECTORY VALIDATION
# ------------------------------------------------------------

if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(
        f"Direktori proyek tidak ditemukan:\n{PROJECT_DIR}"
    )

if not RAW_DATA_DIR.is_dir():
    raise FileNotFoundError(
        f"Direktori data mentah tidak ditemukan:\n{RAW_DATA_DIR}"
    )

if not MAIN_RINEX_FILE.is_file():
    raise FileNotFoundError(
        f"Berkas RINEX utama tidak ditemukan:\n{MAIN_RINEX_FILE}"
    )

# Membuat direktori output jika belum tersedia
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for folder in OUTPUT_SUBDIRS:
    (OUTPUT_DIR / folder).mkdir(exist_ok=True)

# ------------------------------------------------------------
# 5. BASIC INPUT FILE INFORMATION
# ------------------------------------------------------------

file_size_bytes = MAIN_RINEX_FILE.stat().st_size

file_info = {
    "Station": MAIN_STATION,
    "DOY": MAIN_DOY,
    "Filename": MAIN_RINEX_FILE.name,
    "File_Size_Bytes": file_size_bytes,
    "File_Size_MB": round(file_size_bytes / (1024**2), 4),
    "File_Exists": MAIN_RINEX_FILE.is_file(),
    "Output_Directory": str(OUTPUT_DIR)
}

df_file_info = pd.DataFrame([file_info])

# ------------------------------------------------------------
# 6. DISPLAY INITIALIZATION SUMMARY
# ------------------------------------------------------------

print("=" * 65)
print("NOTEBOOK 01 - PROJECT INITIALIZATION")
print("=" * 65)

print(f"Project directory : {PROJECT_DIR}")
print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Main RINEX file   : {MAIN_RINEX_FILE}")
print(f"Output directory  : {OUTPUT_DIR}")

print("\nINITIAL DATASET INFORMATION")
display(df_file_info)

print("\nOUTPUT SUBDIRECTORIES")
for folder in OUTPUT_SUBDIRS:
    print(f"- {folder}")

print("\nInitialization completed successfully.")


NOTEBOOK 01 - PROJECT INITIALIZATION
Project directory : C:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final
Raw data directory: C:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final\data\raw
Main RINEX file   : C:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final\data\raw\DOY300\bako3000.03o
Output directory  : C:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final\Output\Notebook_01

INITIAL DATASET INFORMATION


,Station,DOY,Filename,File_Size_Bytes,File_Size_MB,File_Exists,Output_Directory
0,BAKO,300,bako3000.03o,2103424,2.006,True,C:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data...



OUTPUT SUBDIRECTORIES
- header_metadata
- epoch_structure
- parsed_observations
- diagnostics
- parsing_logs

Initialization completed successfully.


In [17]:
# ============================================================
# STAGE 02 - RINEX HEADER EXTRACTION
# ============================================================

with open(MAIN_RINEX_FILE, "r", encoding="ascii", errors="replace") as file:
    header_lines = []

    for line_number, line in enumerate(file, start=1):
        header_lines.append(line.rstrip("\n\r"))

        if "END OF HEADER" in line:
            header_end_line = line_number
            break

    else:
        raise ValueError("END OF HEADER tidak ditemukan dalam file RINEX.")

# Menampilkan header mentah
print("=" * 75)
print("RINEX HEADER EXTRACTION")
print("=" * 75)

print(f"File                 : {MAIN_RINEX_FILE.name}")
print(f"Jumlah baris header  : {len(header_lines)}")
print(f"END OF HEADER line   : {header_end_line}")
print("\nRAW HEADER CONTENT\n")

for number, line in enumerate(header_lines, start=1):
    print(f"{number:03d} | {line}")

# Pemeriksaan awal struktur header
header_termination = "END OF HEADER" in header_lines[-1]

print("\n" + "=" * 75)
print("HEADER EXTRACTION CHECK")
print("=" * 75)
print(f"Header termination detected : {header_termination}")
print(f"First header line            : {header_lines[0]}")
print(f"Last header line             : {header_lines[-1]}")

if not header_termination:
    raise ValueError("Ekstraksi header belum berhasil.")

RINEX HEADER EXTRACTION
File                 : bako3000.03o
Jumlah baris header  : 18
END OF HEADER line   : 18

RAW HEADER CONTENT

001 |      2.10           OBSERVATION DATA    G (GPS)             RINEX VERSION / TYPE
002 | teqc  2002Mar14                         20031028 12:06:32UTCPGM / RUN BY / DATE
003 | MSWin2000|IAx86-PII|wcl386 11.0|MSWin95/98/NT/2000|486/DX+  COMMENT
004 | BIT 2 OF LLI FLAGS DATA COLLECTED UNDER A/S CONDITION       COMMENT
005 | bako                                                        MARKER NAME
006 | 23101M002                                                   MARKER NUMBER
007 | eka                 bakosurtanal                            OBSERVER / AGENCY
008 | 2491                TRIMBLE 4000SSE     NP 7.19 / SP 3.04   REC # / TYPE / VERS
009 | 44148               TRM14532.00     NONE                    ANT # / TYPE
010 |  -1836967.8873  6065615.4384  -716260.3127                  APPROX POSITION XYZ
011 |         1.6160        0.0000        0.0000     

In [18]:
# ============================================================
# STAGE 03 - HEADER INTERPRETATION
# Fixed-width header field separation
# ============================================================

header_records = []

for line_number, line in enumerate(header_lines, start=1):

    # RINEX header structure:
    # Columns 1-60  : Header content
    # Columns 61-80 : Header label

    content = line[:60].rstrip()
    label = line[60:80].strip()

    header_records.append({
        "Line_Number": line_number,
        "Header_Content": content,
        "Header_Label": label,
        "Line_Length": len(line)
    })

header_table = pd.DataFrame(header_records)

print("=" * 75)
print("RINEX HEADER FIELD INTERPRETATION")
print("=" * 75)

display(header_table)

# ============================================================
# HEADER STRUCTURE DIAGNOSTICS
# ============================================================

print("\n" + "=" * 75)
print("HEADER STRUCTURE DIAGNOSTICS")
print("=" * 75)

print(f"Total header records       : {len(header_table)}")

print("\nHeader line length:")
print(header_table["Line_Length"].value_counts().sort_index())

print("\nHeader label distribution:")
print(
    header_table["Header_Label"]
    .value_counts()
    .to_string()
)

# Check whether header labels are located in columns 61-80
label_check = all(
    record["Header_Label"] == header_lines[
        record["Line_Number"] - 1
    ][60:80].strip()
    for record in header_records
)

print("\nFixed-width label extraction:", label_check)

# Check required header termination
termination_check = (
    header_table.iloc[-1]["Header_Label"] == "END OF HEADER"
)

print("Header termination check   :", termination_check)

RINEX HEADER FIELD INTERPRETATION


,Line_Number,Header_Content,Header_Label,Line_Length
0,1,2.10 OBSERVATION DATA G (GPS),RINEX VERSION / TYPE,80
1,2,teqc 2002Mar14 200310...,PGM / RUN BY / DATE,79
2,3,MSWin2000|IAx86-PII|wcl386 11.0|MSWin95/98/NT/...,COMMENT,67
3,4,BIT 2 OF LLI FLAGS DATA COLLECTED UNDER A/S CO...,COMMENT,67
4,5,bako,MARKER NAME,71
5,6,23101M002,MARKER NUMBER,73
6,7,eka bakosurtanal,OBSERVER / AGENCY,77
7,8,2491 TRIMBLE 4000SSE NP 7.1...,REC # / TYPE / VERS,79
8,9,44148 TRM14532.00 NONE,ANT # / TYPE,72
9,10,-1836967.8873 6065615.4384 -716260.3127,APPROX POSITION XYZ,79



HEADER STRUCTURE DIAGNOSTICS
Total header records       : 18

Header line length:
Line_Length
67    5
71    1
72    1
73    2
77    2
79    4
80    3
Name: count, dtype: int64

Header label distribution:
Header_Label
COMMENT                 5
RINEX VERSION / TYPE    1
PGM / RUN BY / DATE     1
MARKER NAME             1
MARKER NUMBER           1
OBSERVER / AGENCY       1
REC # / TYPE / VERS     1
ANT # / TYPE            1
APPROX POSITION XYZ     1
ANTENNA: DELTA H/E/N    1
WAVELENGTH FACT L1/2    1
# / TYPES OF OBSERV     1
TIME OF FIRST OBS       1
END OF HEADER           1

Fixed-width label extraction: True
Header termination check   : True


In [19]:
# ============================================================
# STAGE 03 - STRUCTURED HEADER METADATA EXTRACTION
# ============================================================

def get_header_content(label):
    """Mengambil isi record berdasarkan label header."""
    matches = [
        line.ljust(80)[:80]
        for line in header_lines
        if line.ljust(80)[60:80].strip() == label
    ]

    if not matches:
        return None

    return matches[0][:60].strip()


# ------------------------------------------------------------
# 1. RINEX VERSION / TYPE
# ------------------------------------------------------------

version_line = next(
    line.ljust(80) for line in header_lines
    if line.ljust(80)[60:80].strip() == "RINEX VERSION / TYPE"
)

rinex_version = version_line[:9].strip()
file_type = version_line[20:40].strip()
satellite_system = version_line[40:60].strip()


# ------------------------------------------------------------
# 2. STATION INFORMATION
# ------------------------------------------------------------

marker_name = get_header_content("MARKER NAME")
marker_number = get_header_content("MARKER NUMBER")

observer_line = next(
    line.ljust(80) for line in header_lines
    if line.ljust(80)[60:80].strip() == "OBSERVER / AGENCY"
)

observer = observer_line[:20].strip()
agency = observer_line[20:40].strip()


# ------------------------------------------------------------
# 3. RECEIVER INFORMATION
# ------------------------------------------------------------

receiver_line = next(
    line.ljust(80) for line in header_lines
    if line.ljust(80)[60:80].strip() == "REC # / TYPE / VERS"
)

receiver_serial = receiver_line[:20].strip()
receiver_type = receiver_line[20:40].strip()
receiver_version = receiver_line[40:60].strip()


# ------------------------------------------------------------
# 4. ANTENNA INFORMATION
# ------------------------------------------------------------

antenna_line = next(
    line.ljust(80) for line in header_lines
    if line.ljust(80)[60:80].strip() == "ANT # / TYPE"
)

antenna_serial = antenna_line[:20].strip()
antenna_type = antenna_line[20:40].strip()
antenna_type_detail = antenna_line[40:60].strip()


# ------------------------------------------------------------
# 5. STATION POSITION
# ------------------------------------------------------------

xyz_values = get_header_content("APPROX POSITION XYZ").split()

station_x, station_y, station_z = map(float, xyz_values)

delta_values = get_header_content(
    "ANTENNA: DELTA H/E/N"
).split()

antenna_height, antenna_east, antenna_north = map(
    float, delta_values
)


# ------------------------------------------------------------
# 6. WAVELENGTH FACTORS
# ------------------------------------------------------------

wavelength_values = get_header_content(
    "WAVELENGTH FACT L1/2"
).split()

wavelength_factor_l1 = int(wavelength_values[0])
wavelength_factor_l2 = int(wavelength_values[1])


# ------------------------------------------------------------
# 7. OBSERVATION TYPES
# ------------------------------------------------------------

obs_line = get_header_content("# / TYPES OF OBSERV")

obs_tokens = obs_line.split()

number_of_observation_types = int(obs_tokens[0])
observation_types = obs_tokens[1:]

if len(observation_types) != number_of_observation_types:
    raise ValueError(
        "Jumlah observation types tidak sesuai dengan header."
    )


# ------------------------------------------------------------
# 8. FIRST OBSERVATION TIME
# ------------------------------------------------------------

time_tokens = get_header_content("TIME OF FIRST OBS").split()

first_obs_year = int(time_tokens[0])
first_obs_month = int(time_tokens[1])
first_obs_day = int(time_tokens[2])
first_obs_hour = int(time_tokens[3])
first_obs_minute = int(time_tokens[4])
first_obs_second = float(time_tokens[5])
time_system = time_tokens[6]


# ------------------------------------------------------------
# 9. BUILD STRUCTURED METADATA
# ------------------------------------------------------------

header_metadata = {
    "RINEX_Version": rinex_version,
    "File_Type": file_type,
    "Satellite_System": satellite_system,
    "Marker_Name": marker_name,
    "Marker_Number": marker_number,
    "Observer": observer,
    "Agency": agency,
    "Receiver_Serial": receiver_serial,
    "Receiver_Type": receiver_type,
    "Receiver_Version": receiver_version,
    "Antenna_Serial": antenna_serial,
    "Antenna_Type": antenna_type,
    "Antenna_Detail": antenna_type_detail,
    "Station_X_m": station_x,
    "Station_Y_m": station_y,
    "Station_Z_m": station_z,
    "Antenna_Height_m": antenna_height,
    "Antenna_East_m": antenna_east,
    "Antenna_North_m": antenna_north,
    "Wavelength_Factor_L1": wavelength_factor_l1,
    "Wavelength_Factor_L2": wavelength_factor_l2,
    "Number_of_Observation_Types": number_of_observation_types,
    "Observation_Types": observation_types,
    "First_Observation_Year": first_obs_year,
    "First_Observation_Month": first_obs_month,
    "First_Observation_Day": first_obs_day,
    "First_Observation_Hour": first_obs_hour,
    "First_Observation_Minute": first_obs_minute,
    "First_Observation_Second": first_obs_second,
    "Time_System": time_system
}


# ------------------------------------------------------------
# 10. DISPLAY RESULTS
# ------------------------------------------------------------

metadata_table = pd.DataFrame(
    list(header_metadata.items()),
    columns=["Metadata_Field", "Metadata_Value"]
)

print("=" * 75)
print("STRUCTURED RINEX HEADER METADATA")
print("=" * 75)

display(metadata_table)

print("\nObservation types:", observation_types)
print("Number of observation types:", len(observation_types))
print("Time system:", time_system)

STRUCTURED RINEX HEADER METADATA


,Metadata_Field,Metadata_Value
0,RINEX_Version,2.10
1,File_Type,OBSERVATION DATA
2,Satellite_System,G (GPS)
3,Marker_Name,bako
4,Marker_Number,23101M002
5,Observer,eka
6,Agency,bakosurtanal
7,Receiver_Serial,2491
8,Receiver_Type,TRIMBLE 4000SSE
9,Receiver_Version,NP 7.19 / SP 3.04



Observation types: ['L1', 'L2', 'C1', 'P2', 'P1', 'D1']
Number of observation types: 6
Time system: GPS


In [20]:
# ============================================================
# STAGE 03 - METADATA VALIDATION
# ============================================================

expected_metadata = {
    "RINEX_Version": "2.10",
    "File_Type": "OBSERVATION DATA",
    "Satellite_System": "G (GPS)",
    "Marker_Name": "bako",
    "Marker_Number": "23101M002",
    "Receiver_Serial": "2491",
    "Receiver_Type": "TRIMBLE 4000SSE",
    "Receiver_Version": "NP 7.19 / SP 3.04",
    "Antenna_Serial": "44148",
    "Antenna_Type": "TRM14532.00",
    "Station_X_m": -1836967.8873,
    "Station_Y_m": 6065615.4384,
    "Station_Z_m": -716260.3127,
    "Antenna_Height_m": 1.6160,
    "Antenna_East_m": 0.0,
    "Antenna_North_m": 0.0,
    "Wavelength_Factor_L1": 1,
    "Wavelength_Factor_L2": 1,
    "Number_of_Observation_Types": 6,
    "Observation_Types": ["L1", "L2", "C1", "P2", "P1", "D1"],
    "First_Observation_Year": 2003,
    "First_Observation_Month": 10,
    "First_Observation_Day": 27,
    "First_Observation_Hour": 0,
    "First_Observation_Minute": 0,
    "First_Observation_Second": 30.0,
    "Time_System": "GPS"
}

validation_results = []

for field, expected in expected_metadata.items():

    actual = header_metadata[field]

    if isinstance(expected, float):
        passed = np.isclose(actual, expected, atol=1e-8)
    else:
        passed = actual == expected

    validation_results.append({
        "Metadata_Field": field,
        "Expected_Value": str(expected),
        "Extracted_Value": str(actual),
        "Status": "PASS" if passed else "FAIL"
    })

validation_table = pd.DataFrame(validation_results)

display(validation_table)

print("\nVALIDATION SUMMARY")
print("Total fields :", len(validation_table))
print("PASS         :", (validation_table["Status"] == "PASS").sum())
print("FAIL         :", (validation_table["Status"] == "FAIL").sum())

if (validation_table["Status"] == "FAIL").any():
    print("\nMetadata validation requires investigation.")
else:
    print("\nAll selected metadata fields match the reference header.")

,Metadata_Field,Expected_Value,Extracted_Value,Status
0,RINEX_Version,2.10,2.10,PASS
1,File_Type,OBSERVATION DATA,OBSERVATION DATA,PASS
2,Satellite_System,G (GPS),G (GPS),PASS
3,Marker_Name,bako,bako,PASS
4,Marker_Number,23101M002,23101M002,PASS
5,Receiver_Serial,2491,2491,PASS
6,Receiver_Type,TRIMBLE 4000SSE,TRIMBLE 4000SSE,PASS
7,Receiver_Version,NP 7.19 / SP 3.04,NP 7.19 / SP 3.04,PASS
8,Antenna_Serial,44148,44148,PASS
9,Antenna_Type,TRM14532.00,TRM14532.00 NONE,FAIL



VALIDATION SUMMARY
Total fields : 27
PASS         : 26
FAIL         : 1

Metadata validation requires investigation.


In [21]:
# ============================================================
# STAGE 03 - CORRECTED METADATA VALIDATION
# ============================================================

expected_metadata["Antenna_Type"] = "TRM14532.00     NONE"

validation_results = []

for field, expected in expected_metadata.items():

    actual = header_metadata[field]

    if isinstance(expected, float):
        passed = np.isclose(actual, expected, atol=1e-8)
    else:
        passed = actual == expected

    validation_results.append({
        "Metadata_Field": field,
        "Expected_Value": str(expected),
        "Extracted_Value": str(actual),
        "Status": "PASS" if passed else "FAIL"
    })

validation_table = pd.DataFrame(validation_results)

display(validation_table)

print("\nCORRECTED VALIDATION SUMMARY")
print("Total fields :", len(validation_table))
print("PASS         :", (validation_table["Status"] == "PASS").sum())
print("FAIL         :", (validation_table["Status"] == "FAIL").sum())

if (validation_table["Status"] == "FAIL").any():
    print("\nMetadata validation requires further investigation.")
else:
    print("\nAll selected metadata fields match the reference header.")

,Metadata_Field,Expected_Value,Extracted_Value,Status
0,RINEX_Version,2.10,2.10,PASS
1,File_Type,OBSERVATION DATA,OBSERVATION DATA,PASS
2,Satellite_System,G (GPS),G (GPS),PASS
3,Marker_Name,bako,bako,PASS
4,Marker_Number,23101M002,23101M002,PASS
5,Receiver_Serial,2491,2491,PASS
6,Receiver_Type,TRIMBLE 4000SSE,TRIMBLE 4000SSE,PASS
7,Receiver_Version,NP 7.19 / SP 3.04,NP 7.19 / SP 3.04,PASS
8,Antenna_Serial,44148,44148,PASS
9,Antenna_Type,TRM14532.00 NONE,TRM14532.00 NONE,PASS



CORRECTED VALIDATION SUMMARY
Total fields : 27
PASS         : 27
FAIL         : 0

All selected metadata fields match the reference header.


In [22]:
# ============================================================
# STAGE 03 - SAVE STRUCTURED HEADER METADATA
# ============================================================

import json

# Output paths
HEADER_OUTPUT_DIR = OUTPUT_DIR / "header_metadata"

metadata_csv_path = (
    HEADER_OUTPUT_DIR / "header_metadata_doy300.csv"
)

metadata_json_path = (
    HEADER_OUTPUT_DIR / "header_metadata_doy300.json"
)

validation_csv_path = (
    HEADER_OUTPUT_DIR / "header_metadata_validation_doy300.csv"
)

# 1. Save metadata as CSV
metadata_table.to_csv(
    metadata_csv_path,
    index=False
)

# 2. Save metadata as JSON
metadata_document = {
    "source_file": MAIN_RINEX_FILE.name,
    "station": MAIN_STATION,
    "doy": MAIN_DOY,
    "metadata": header_metadata
}

with open(metadata_json_path, "w", encoding="utf-8") as file:
    json.dump(
        metadata_document,
        file,
        indent=4,
        ensure_ascii=False
    )

# 3. Save validation results
validation_table.to_csv(
    validation_csv_path,
    index=False
)

# 4. Verify saved outputs
output_files = [
    metadata_csv_path,
    metadata_json_path,
    validation_csv_path
]

print("=" * 75)
print("STAGE 03 - METADATA EXPORT REPORT")
print("=" * 75)

for path in output_files:
    print(f"\nFile       : {path.name}")
    print(f"Exists     : {path.exists()}")
    print(f"Size       : {path.stat().st_size if path.exists() else 0} bytes")

# 5. Read-back verification
with open(metadata_json_path, "r", encoding="utf-8") as file:
    metadata_readback = json.load(file)

json_check = (
    metadata_readback["metadata"] == header_metadata
)

csv_check = pd.read_csv(metadata_csv_path).shape[0] == len(
    header_metadata
)

validation_check = (
    pd.read_csv(validation_csv_path)["Status"].eq("PASS").all()
)

print("\n" + "=" * 75)
print("EXPORT VERIFICATION")
print("=" * 75)

print("JSON read-back consistency :", json_check)
print("CSV record count           :", csv_check)
print("Validation export          :", validation_check)

if all([json_check, csv_check, validation_check]):
    print("\nSTAGE 03 EXPORT SUCCESSFUL.")
else:
    print("\nSTAGE 03 EXPORT REQUIRES INVESTIGATION.")


STAGE 03 - METADATA EXPORT REPORT

File       : header_metadata_doy300.csv
Exists     : True
Size       : 812 bytes

File       : header_metadata_doy300.json
Exists     : True
Size       : 1362 bytes

File       : header_metadata_validation_doy300.csv
Exists     : True
Size       : 1152 bytes

EXPORT VERIFICATION
JSON read-back consistency : True
CSV record count           : True
Validation export          : True

STAGE 03 EXPORT SUCCESSFUL.


In [23]:
# ============================================================
# STAGE 04 - EPOCH RECORD STRUCTURE ANALYSIS
# Initial epoch record extraction
# ============================================================

import re

# ------------------------------------------------------------
# 1. READ RAW RINEX FILE
# ------------------------------------------------------------

with open(MAIN_RINEX_FILE, "r", encoding="ascii", errors="replace") as file:
    rinex_lines = file.readlines()

# Header telah diketahui berakhir pada header_end_line.
# Python menggunakan zero-based indexing.
data_start_index = header_end_line

observation_lines = rinex_lines[data_start_index:]

print("=" * 75)
print("EPOCH RECORD STRUCTURE ANALYSIS")
print("=" * 75)

print(f"Total physical lines in file : {len(rinex_lines)}")
print(f"Header lines                 : {header_end_line}")
print(f"Observation section lines    : {len(observation_lines)}")


# ------------------------------------------------------------
# 2. INITIAL EPOCH RECORD EXTRACTION
# ------------------------------------------------------------

epoch_records = []

for line_index, raw_line in enumerate(
    observation_lines,
    start=data_start_index + 1
):

    line = raw_line.rstrip("\r\n").ljust(80)

    # RINEX 2.10 epoch record:
    # Columns  1-2  : Year
    # Columns  4-5  : Month
    # Columns  7-8  : Day
    # Columns 10-11 : Hour
    # Columns 13-14 : Minute
    # Columns 16-26 : Second
    # Column  29    : Epoch flag
    # Columns 30-32 : Number of satellites
    # Columns 33-68 : Satellite identifiers (first line)

    try:
        yy = int(line[1:3])
        month = int(line[4:6])
        day = int(line[7:9])
        hour = int(line[10:12])
        minute = int(line[13:15])
        second = float(line[15:26])

        epoch_flag = int(line[28:29])
        number_of_satellites = int(line[29:32])

    except (ValueError, TypeError):
        continue

    # Basic range checks
    if not (
        1 <= month <= 12
        and 1 <= day <= 31
        and 0 <= hour <= 23
        and 0 <= minute <= 59
        and 0 <= second < 61
        and 0 <= epoch_flag <= 6
        and number_of_satellites >= 0
    ):
        continue

    # RINEX 2.10 two-digit year interpretation
    year = 2000 + yy if yy < 80 else 1900 + yy

    try:
        epoch_time = (
            pd.Timestamp(
                year=year,
                month=month,
                day=day,
                hour=hour,
                minute=minute
            )
            + pd.Timedelta(seconds=second)
        )

    except ValueError:
        continue

    # Satellite identifiers on the first epoch line.
    satellite_text = line[32:68]

    satellite_ids = [
        satellite_text[i:i + 3].strip()
        for i in range(0, 36, 3)
        if satellite_text[i:i + 3].strip()
    ]

    epoch_records.append({
        "Physical_Line_Number": line_index,
        "Epoch_Time_GPS": epoch_time,
        "Epoch_Flag": epoch_flag,
        "Declared_Satellite_Count": number_of_satellites,
        "Satellite_IDs_First_Line": satellite_ids,
        "Satellite_Count_First_Line": len(satellite_ids),
        "Physical_Line_Length": len(raw_line.rstrip("\r\n"))
    })


# ------------------------------------------------------------
# 3. CREATE EPOCH DATAFRAME
# ------------------------------------------------------------

df_epochs = pd.DataFrame(epoch_records)

if df_epochs.empty:
    raise ValueError(
        "Tidak ada epoch record yang berhasil diekstrak. "
        "Periksa kembali posisi kolom RINEX."
    )

print("\n" + "=" * 75)
print("EPOCH EXTRACTION SUMMARY")
print("=" * 75)

print(f"Extracted epoch records : {len(df_epochs)}")
print(f"First epoch             : {df_epochs['Epoch_Time_GPS'].min()}")
print(f"Last epoch              : {df_epochs['Epoch_Time_GPS'].max()}")
print(f"Unique epoch timestamps : {df_epochs['Epoch_Time_GPS'].nunique()}")


# ------------------------------------------------------------
# 4. DISPLAY INITIAL RECORDS
# ------------------------------------------------------------

print("\nFIRST 10 EPOCH RECORDS")

display(df_epochs.head(10))


# ------------------------------------------------------------
# 5. EPOCH FLAG DISTRIBUTION
# ------------------------------------------------------------

print("\nEPOCH FLAG DISTRIBUTION")

display(
    df_epochs["Epoch_Flag"]
    .value_counts()
    .sort_index()
    .rename_axis("Epoch_Flag")
    .reset_index(name="Count")
)


# ------------------------------------------------------------
# 6. SATELLITE COUNT COMPARISON
# ------------------------------------------------------------

df_epochs["Satellite_Count_Match_First_Line"] = (
    df_epochs["Declared_Satellite_Count"]
    == df_epochs["Satellite_Count_First_Line"]
)

print("\nSATELLITE COUNT COMPARISON")

print(
    "Declared count matches first-line count:",
    df_epochs["Satellite_Count_Match_First_Line"].sum(),
    "/",
    len(df_epochs)
)

print(
    "Potential continuation records:",
    (~df_epochs["Satellite_Count_Match_First_Line"]).sum()
)

EPOCH RECORD STRUCTURE ANALYSIS
Total physical lines in file : 51404
Header lines                 : 18
Observation section lines    : 51386

EPOCH EXTRACTION SUMMARY
Extracted epoch records : 2878
First epoch             : 2003-10-27 00:00:30
Last epoch              : 2003-10-27 23:59:00.128000
Unique epoch timestamps : 2878

FIRST 10 EPOCH RECORDS


,Physical_Line_Number,Epoch_Time_GPS,Epoch_Flag,Declared_Satellite_Count,Satellite_IDs_First_Line,Satellite_Count_First_Line,Physical_Line_Length
0,19,2003-10-27 00:00:30.000,0,9,"[G21, G 6, G 5, G25, G15, G18, G30, G14, G23]",9,59
1,38,2003-10-27 00:01:00.000,0,9,"[G21, G 6, G 5, G25, G15, G18, G30, G14, G23]",9,59
2,57,2003-10-27 00:01:30.000,0,9,"[G21, G 6, G 5, G25, G15, G18, G30, G14, G23]",9,59
3,76,2003-10-27 00:02:00.001,0,9,"[G21, G 6, G 5, G25, G15, G18, G30, G14, G23]",9,59
4,95,2003-10-27 00:02:30.001,0,9,"[G21, G 6, G 5, G25, G15, G18, G30, G14, G23]",9,59
5,114,2003-10-27 00:03:00.001,0,9,"[G21, G 6, G 5, G25, G15, G18, G30, G14, G23]",9,59
6,133,2003-10-27 00:03:30.001,0,9,"[G21, G 6, G 5, G25, G15, G18, G30, G14, G23]",9,59
7,152,2003-10-27 00:04:00.001,0,9,"[G21, G 6, G 5, G25, G15, G18, G30, G14, G23]",9,59
8,171,2003-10-27 00:04:30.001,0,9,"[G21, G 6, G 5, G25, G15, G18, G30, G14, G23]",9,59
9,190,2003-10-27 00:05:00.001,0,9,"[G21, G 6, G 5, G25, G15, G18, G30, G14, G23]",9,59



EPOCH FLAG DISTRIBUTION


,Epoch_Flag,Count
0,0,2878



SATELLITE COUNT COMPARISON
Declared count matches first-line count: 2878 / 2878
Potential continuation records: 0


In [24]:
# ============================================================
# STAGE 04 - EPOCH STRUCTURE QUALITY CONTROL
# ============================================================

# ------------------------------------------------------------
# 1. NORMALIZE SATELLITE IDENTIFIERS
# ------------------------------------------------------------

def normalize_satellite_id(satellite_id):

    satellite_id = satellite_id.strip()

    if not satellite_id:
        return None

    system = satellite_id[0]
    prn_text = satellite_id[1:].strip()

    if not prn_text.isdigit():
        return satellite_id

    return f"{system}{int(prn_text):02d}"


df_epochs["Satellite_IDs_Normalized"] = (
    df_epochs["Satellite_IDs_First_Line"]
    .apply(
        lambda satellites: [
            normalize_satellite_id(sat)
            for sat in satellites
        ]
    )
)

# ------------------------------------------------------------
# 2. TEMPORAL SEQUENCE VALIDATION
# ------------------------------------------------------------

df_epochs = df_epochs.sort_values(
    "Physical_Line_Number"
).reset_index(drop=True)

df_epochs["Time_Difference_s"] = (
    df_epochs["Epoch_Time_GPS"].diff().dt.total_seconds()
)

interval_distribution = (
    df_epochs["Time_Difference_s"]
    .dropna()
    .round(3)
    .value_counts()
    .sort_index()
)

chronological_check = (
    df_epochs["Epoch_Time_GPS"].is_monotonic_increasing
)

duplicate_timestamp_count = (
    df_epochs["Epoch_Time_GPS"].duplicated().sum()
)

# ------------------------------------------------------------
# 3. SATELLITE COUNT VALIDATION
# ------------------------------------------------------------

df_epochs["Normalized_Satellite_Count"] = (
    df_epochs["Satellite_IDs_Normalized"].apply(len)
)

df_epochs["Satellite_Count_Consistent"] = (
    df_epochs["Declared_Satellite_Count"]
    == df_epochs["Normalized_Satellite_Count"]
)

satellite_count_mismatch = (
    ~df_epochs["Satellite_Count_Consistent"]
).sum()

# ------------------------------------------------------------
# 4. SATELLITE IDENTITY VALIDATION
# ------------------------------------------------------------

all_satellites = sorted(
    set(
        satellite
        for satellites in df_epochs["Satellite_IDs_Normalized"]
        for satellite in satellites
    )
)

duplicate_satellite_epochs = sum(
    len(satellites) != len(set(satellites))
    for satellites in df_epochs["Satellite_IDs_Normalized"]
)

# ------------------------------------------------------------
# 5. DIAGNOSTIC REPORT
# ------------------------------------------------------------

print("=" * 75)
print("EPOCH STRUCTURE QUALITY CONTROL")
print("=" * 75)

print("\nTEMPORAL VALIDATION")
print("Chronological order       :", chronological_check)
print("Duplicate timestamps      :", duplicate_timestamp_count)
print("Median interval (s)       :", df_epochs["Time_Difference_s"].median())
print("Minimum interval (s)      :", df_epochs["Time_Difference_s"].min())
print("Maximum interval (s)      :", df_epochs["Time_Difference_s"].max())

print("\nINTERVAL DISTRIBUTION")
display(
    interval_distribution.rename_axis("Interval_s")
    .reset_index(name="Count")
)

print("\nSATELLITE VALIDATION")
print("Unique satellite IDs      :", len(all_satellites))
print("Satellite IDs             :", all_satellites)
print("Satellite count mismatch  :", satellite_count_mismatch)
print("Duplicate IDs within epoch:", duplicate_satellite_epochs)

print("\nNORMALIZED SATELLITE SAMPLE")
display(
    df_epochs[
        [
            "Epoch_Time_GPS",
            "Declared_Satellite_Count",
            "Satellite_IDs_Normalized"
        ]
    ].head(5)
)

EPOCH STRUCTURE QUALITY CONTROL

TEMPORAL VALIDATION
Chronological order       : True
Duplicate timestamps      : 0
Median interval (s)       : 30.0
Minimum interval (s)      : 30.0
Maximum interval (s)      : 30.001

INTERVAL DISTRIBUTION


,Interval_s,Count
0,30.000,2749
1,30.001,128



SATELLITE VALIDATION
Unique satellite IDs      : 28
Satellite IDs             : ['G01', 'G02', 'G03', 'G04', 'G05', 'G06', 'G07', 'G08', 'G09', 'G10', 'G11', 'G13', 'G14', 'G15', 'G16', 'G17', 'G18', 'G20', 'G21', 'G23', 'G24', 'G25', 'G26', 'G27', 'G28', 'G29', 'G30', 'G31']
Satellite count mismatch  : 0
Duplicate IDs within epoch: 0

NORMALIZED SATELLITE SAMPLE


,Epoch_Time_GPS,Declared_Satellite_Count,Satellite_IDs_Normalized
0,2003-10-27 00:00:30.000,9,"[G21, G06, G05, G25, G15, G18, G30, G14, G23]"
1,2003-10-27 00:01:00.000,9,"[G21, G06, G05, G25, G15, G18, G30, G14, G23]"
2,2003-10-27 00:01:30.000,9,"[G21, G06, G05, G25, G15, G18, G30, G14, G23]"
3,2003-10-27 00:02:00.001,9,"[G21, G06, G05, G25, G15, G18, G30, G14, G23]"
4,2003-10-27 00:02:30.001,9,"[G21, G06, G05, G25, G15, G18, G30, G14, G23]"


In [25]:
# Menampilkan variabel DataFrame yang tersedia
for name, value in globals().copy().items():
    if isinstance(value, pd.DataFrame):
        print(f"{name}: {value.shape}")

df_file_info: (1, 7)
header_table: (18, 4)
metadata_table: (30, 2)
validation_table: (27, 4)
df_epochs: (2878, 12)
epoch_structure_qc: (2878, 12)


In [26]:
# ============================================================
# EXPORT: EPOCH STRUCTURE QUALITY CONTROL
# ============================================================

epoch_structure_output = (
    OUTPUT_DIR
    / "epoch_structure"
    / "epoch_structure_qc_doy300.csv"
)

epoch_structure_qc = df_epochs.copy()

epoch_structure_qc.to_csv(
    epoch_structure_output,
    index=False
)

print("EPOCH STRUCTURE QC EXPORT")
print("=" * 70)
print(f"Output file : {epoch_structure_output}")
print(f"File exists : {epoch_structure_output.exists()}")
print(f"Total rows  : {len(epoch_structure_qc)}")
print(f"File size   : {epoch_structure_output.stat().st_size:,} bytes")

EPOCH STRUCTURE QC EXPORT
Output file : C:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final\Output\Notebook_01\epoch_structure\epoch_structure_qc_doy300.csv
File exists : True
Total rows  : 2878
File size   : 520,803 bytes


In [27]:
# ============================================================
# STAGE 05 — SATELLITE OBSERVATION RECORD PARSING
# STEP 1: RAW OBSERVATION RECORD INSPECTION
# ============================================================

with open(MAIN_RINEX_FILE, "r", errors="replace") as file:
    raw_lines = file.readlines()

# Header berakhir pada END OF HEADER
header_end_index = next(
    i for i, line in enumerate(raw_lines)
    if line[60:80].strip() == "END OF HEADER"
)

# Ambil 15 baris pertama setelah header
observation_sample = raw_lines[
    header_end_index + 1 : header_end_index + 16
]

print("RAW OBSERVATION RECORD INSPECTION")
print("=" * 80)
print(f"Header ends at line : {header_end_index + 1}")
print(f"First data line     : {header_end_index + 2}")
print(f"Sample lines        : {len(observation_sample)}")

for i, line in enumerate(observation_sample, start=1):
    print(f"\nLine {i}")
    print(f"Length : {len(line.rstrip())}")
    print(f"Raw    : {repr(line.rstrip())}")

RAW OBSERVATION RECORD INSPECTION
Header ends at line : 18
First data line     : 19
Sample lines        : 15

Line 1
Length : 59
Raw    : ' 03 10 27  0  0 30.0000000  0  9G21G 6G 5G25G15G18G30G14G23'

Line 2
Length : 63
Raw    : '   7272414.07557   5719375.47357  20659955.1484   20659955.7704'

Line 3
Length : 15
Raw    : '     -3921.0944'

Line 4
Length : 63
Raw    : '  35449963.44555   2632699.29255  23215464.2584   23215469.9774'

Line 5
Length : 15
Raw    : '     -3146.2974'

Line 6
Length : 63
Raw    : '  56683781.12656  17932985.92856  22960277.9694   22960283.2304'

Line 7
Length : 15
Raw    : '     -3966.5624'

Line 8
Length : 63
Raw    : '   1091063.23856    872713.18356  22358403.5784   22358406.5554'

Line 9
Length : 15
Raw    : '     -1224.5944'

Line 10
Length : 63
Raw    : '  -3502904.17657  -2648798.11957  21209930.3134   21209934.1454'

Line 11
Length : 15
Raw    : '      -875.0784'

Line 12
Length : 63
Raw    : '   3471119.70456   1710390.59356  22159258.9774   2215926

In [28]:
# ============================================================
# STAGE 05 — STEP 2
# FIXED-WIDTH OBSERVATION FIELD INSPECTION
# ============================================================

print("FIXED-WIDTH FIELD INSPECTION")
print("=" * 80)

# Periksa lima baris fisik pertama setelah epoch record
for line_number, raw_line in enumerate(
    raw_lines[header_end_index + 2 : header_end_index + 7],
    start=header_end_index + 3
):
    line = raw_line.rstrip("\r\n")

    print(f"\nPhysical line : {line_number}")
    print(f"Original length: {len(line)}")
    print(f"Raw content    : {repr(line)}")

    # Padding hanya untuk inspeksi posisi field,
    # bukan mengubah data mentah.
    padded_line = line.ljust(80)

    for field_index in range(5):
        start = field_index * 16
        end = start + 16
        field = padded_line[start:end]

        print(
            f"Field {field_index + 1} "
            f"[{start}:{end}] : {repr(field)}"
        )

FIXED-WIDTH FIELD INSPECTION

Physical line : 20
Original length: 63
Raw content    : '   7272414.07557   5719375.47357  20659955.1484   20659955.7704'
Field 1 [0:16] : '   7272414.07557'
Field 2 [16:32] : '   5719375.47357'
Field 3 [32:48] : '  20659955.1484 '
Field 4 [48:64] : '  20659955.7704 '
Field 5 [64:80] : '                '

Physical line : 21
Original length: 15
Raw content    : '     -3921.0944'
Field 1 [0:16] : '     -3921.0944 '
Field 2 [16:32] : '                '
Field 3 [32:48] : '                '
Field 4 [48:64] : '                '
Field 5 [64:80] : '                '

Physical line : 22
Original length: 63
Raw content    : '  35449963.44555   2632699.29255  23215464.2584   23215469.9774'
Field 1 [0:16] : '  35449963.44555'
Field 2 [16:32] : '   2632699.29255'
Field 3 [32:48] : '  23215464.2584 '
Field 4 [48:64] : '  23215469.9774 '
Field 5 [64:80] : '                '

Physical line : 23
Original length: 15
Raw content    : '     -3146.2974'
Field 1 [0:16] : '     

In [29]:
# ============================================================
# STAGE 05 — STEP 3
# TWO-LINE SATELLITE RECORD RECONSTRUCTION
# ============================================================

print("SATELLITE OBSERVATION RECORD RECONSTRUCTION")
print("=" * 80)

# Dua baris pertama setelah epoch record
line_1 = raw_lines[header_end_index + 1].rstrip("\r\n")
line_2 = raw_lines[header_end_index + 2].rstrip("\r\n")

# Padding untuk mempertahankan posisi fixed-width
line_1 = line_1.ljust(80)
line_2 = line_2.ljust(80)

# RINEX 2.x: maksimal lima field observasi pada satu baris
combined_record = line_1 + line_2[:16]

observation_types = ["L1", "L2", "C1", "P2", "P1", "D1"]

print("Physical line 1:", repr(line_1.rstrip()))
print("Physical line 2:", repr(line_2.rstrip()))
print("\nRECONSTRUCTED OBSERVATION FIELDS")
print("-" * 80)

for i, obs_type in enumerate(observation_types):
    start = i * 16
    field = combined_record[start:start + 16]

    value_raw = field[:14]
    lli = field[14]
    ssi = field[15]

    value = pd.to_numeric(
        value_raw.strip(),
        errors="coerce"
    )

    print(f"\n{obs_type}")
    print(f"Raw field : {repr(field)}")
    print(f"Value     : {value}")
    print(f"LLI       : {repr(lli)}")
    print(f"SSI       : {repr(ssi)}")

SATELLITE OBSERVATION RECORD RECONSTRUCTION
Physical line 1: ' 03 10 27  0  0 30.0000000  0  9G21G 6G 5G25G15G18G30G14G23'
Physical line 2: '   7272414.07557   5719375.47357  20659955.1484   20659955.7704'

RECONSTRUCTED OBSERVATION FIELDS
--------------------------------------------------------------------------------

L1
Raw field : ' 03 10 27  0  0 '
Value     : nan
LLI       : '0'
SSI       : ' '

L2
Raw field : '30.0000000  0  9'
Value     : nan
LLI       : ' '
SSI       : '9'

C1
Raw field : 'G21G 6G 5G25G15G'
Value     : nan
LLI       : '5'
SSI       : 'G'

P2
Raw field : '18G30G14G23     '
Value     : nan
LLI       : ' '
SSI       : ' '

P1
Raw field : '                '
Value     : nan
LLI       : ' '
SSI       : ' '

D1
Raw field : '   7272414.07557'
Value     : 7272414.075
LLI       : '5'
SSI       : '7'


In [30]:
# ============================================================
# STAGE 05 — STEP 3
# SATELLITE OBSERVATION RECORD STRUCTURE INSPECTION
# ============================================================

# Epoch pertama berada pada baris indeks 18.
# Dua baris berikutnya merupakan kandidat record observasi G21.

candidate_lines = raw_lines[19:21]

print("SATELLITE OBSERVATION RECORD INSPECTION")
print("=" * 80)

for i, line in enumerate(candidate_lines, start=1):
    clean_line = line.rstrip("\r\n")

    print(f"\nObservation line {i}")
    print(f"Length : {len(clean_line)}")
    print(f"Raw    : {repr(clean_line)}")

    padded_line = clean_line.ljust(80)

    for field_index in range(5):
        start = field_index * 16
        end = start + 16
        field = padded_line[start:end]

        print(
            f"Field {field_index + 1} "
            f"[{start}:{end}] : {repr(field)}"
        )

SATELLITE OBSERVATION RECORD INSPECTION

Observation line 1
Length : 63
Raw    : '   7272414.07557   5719375.47357  20659955.1484   20659955.7704'
Field 1 [0:16] : '   7272414.07557'
Field 2 [16:32] : '   5719375.47357'
Field 3 [32:48] : '  20659955.1484 '
Field 4 [48:64] : '  20659955.7704 '
Field 5 [64:80] : '                '

Observation line 2
Length : 15
Raw    : '     -3921.0944'
Field 1 [0:16] : '     -3921.0944 '
Field 2 [16:32] : '                '
Field 3 [32:48] : '                '
Field 4 [48:64] : '                '
Field 5 [64:80] : '                '


In [31]:
# ============================================================
# STAGE 05 — STEP 4
# CORRECTED OBSERVATION FIELD RECONSTRUCTION
# ============================================================

observation_types = ["L1", "L2", "C1", "P2", "P1", "D1"]

# Dua baris fisik record observasi G21
satellite_lines = [
    raw_lines[19].rstrip("\r\n"),
    raw_lines[20].rstrip("\r\n")
]

# Baris pertama menyediakan lima slot field.
# Baris kedua melanjutkan field keenam.
first_line = satellite_lines[0].ljust(80)
continuation_line = satellite_lines[1].ljust(80)

raw_fields = []

# Lima slot pertama dari baris pertama
for i in range(5):
    raw_fields.append(first_line[i * 16:(i + 1) * 16])

# Field keenam dari baris lanjutan
raw_fields.append(continuation_line[0:16])

reconstructed_records = []

for obs_type, field in zip(observation_types, raw_fields):

    value_text = field[:14].strip()
    lli = field[14]
    ssi = field[15]

    try:
        value = float(value_text) if value_text else np.nan
    except ValueError:
        value = np.nan

    reconstructed_records.append({
        "Observation_Type": obs_type,
        "Raw_Field": repr(field),
        "Value": value,
        "LLI": lli,
        "SSI": ssi
    })

df_reconstructed_g21 = pd.DataFrame(reconstructed_records)

print("CORRECTED SATELLITE OBSERVATION RECONSTRUCTION")
print("=" * 80)

display(df_reconstructed_g21)

CORRECTED SATELLITE OBSERVATION RECONSTRUCTION


,Observation_Type,Raw_Field,Value,LLI,SSI
0,L1,' 7272414.07557',7.272414e+06,5,7
1,L2,' 5719375.47357',5.719375e+06,5,7
2,C1,' 20659955.1484 ',2.065996e+07,4,
3,P2,' 20659955.7704 ',2.065996e+07,4,
4,P1,' ',NaN,,
5,D1,' -3921.0944 ',-3.921094e+03,4,


In [32]:
# ============================================================
# STAGE 05 — STEP 5
# OBSERVATION RECORD CONTINUATION CHECK
# ============================================================

print("OBSERVATION RECORD CONTINUATION CHECK")
print("=" * 80)

# Baris fisik 22–27
for line_number, line in enumerate(raw_lines[21:27], start=22):

    clean_line = line.rstrip("\r\n")
    padded_line = clean_line.ljust(80)

    nonempty_fields = [
        padded_line[i:i+16]
        for i in range(0, 80, 16)
        if padded_line[i:i+16].strip()
    ]

    print(f"\nPhysical line : {line_number}")
    print(f"Length        : {len(clean_line)}")
    print(f"Raw           : {repr(clean_line)}")
    print(f"Non-empty fields: {len(nonempty_fields)}")

    for i, field in enumerate(nonempty_fields, start=1):
        print(f"  Field {i}: {repr(field)}")

OBSERVATION RECORD CONTINUATION CHECK

Physical line : 22
Length        : 63
Raw           : '  35449963.44555   2632699.29255  23215464.2584   23215469.9774'
Non-empty fields: 4
  Field 1: '  35449963.44555'
  Field 2: '   2632699.29255'
  Field 3: '  23215464.2584 '
  Field 4: '  23215469.9774 '

Physical line : 23
Length        : 15
Raw           : '     -3146.2974'
Non-empty fields: 1
  Field 1: '     -3146.2974 '

Physical line : 24
Length        : 63
Raw           : '  56683781.12656  17932985.92856  22960277.9694   22960283.2304'
Non-empty fields: 4
  Field 1: '  56683781.12656'
  Field 2: '  17932985.92856'
  Field 3: '  22960277.9694 '
  Field 4: '  22960283.2304 '

Physical line : 25
Length        : 15
Raw           : '     -3966.5624'
Non-empty fields: 1
  Field 1: '     -3966.5624 '

Physical line : 26
Length        : 63
Raw           : '   1091063.23856    872713.18356  22358403.5784   22358406.5554'
Non-empty fields: 4
  Field 1: '   1091063.23856'
  Field 2: '    872713.

In [33]:
# ============================================================
# STAGE 05 — STEP 6
# FIRST EPOCH RECORD BOUNDARY VALIDATION
# ============================================================

n_satellites = 9
n_observations = len(observation_types)

# Setiap satelit memiliki 6 field.
# Kapasitas satu baris observasi adalah 5 field.
# Baris lanjutan menyediakan field sisanya.
physical_lines_per_satellite = (
    (n_observations + 4) // 5
)

print("FIRST EPOCH RECORD BOUNDARY VALIDATION")
print("=" * 80)
print(f"Declared satellites       : {n_satellites}")
print(f"Observation types         : {n_observations}")
print(f"Physical lines per record : {physical_lines_per_satellite}")
print(
    f"Expected observation lines: "
    f"{n_satellites * physical_lines_per_satellite}"
)

# Baris observasi pertama dimulai setelah epoch record.
first_observation_index = header_end_index + 2

for sat_index in range(n_satellites):

    start = (
        first_observation_index
        + sat_index * physical_lines_per_satellite
    )

    record_lines = raw_lines[
        start:start + physical_lines_per_satellite
    ]

    print(f"\nSatellite position {sat_index + 1}")
    for line in record_lines:
        print(repr(line.rstrip("\r\n")))

FIRST EPOCH RECORD BOUNDARY VALIDATION
Declared satellites       : 9
Observation types         : 6
Physical lines per record : 2
Expected observation lines: 18

Satellite position 1
'   7272414.07557   5719375.47357  20659955.1484   20659955.7704'
'     -3921.0944'

Satellite position 2
'  35449963.44555   2632699.29255  23215464.2584   23215469.9774'
'     -3146.2974'

Satellite position 3
'  56683781.12656  17932985.92856  22960277.9694   22960283.2304'
'     -3966.5624'

Satellite position 4
'   1091063.23856    872713.18356  22358403.5784   22358406.5554'
'     -1224.5944'

Satellite position 5
'  -3502904.17657  -2648798.11957  21209930.3134   21209934.1454'
'      -875.0784'

Satellite position 6
'   3471119.70456   1710390.59356  22159258.9774   22159262.7154'
'      -272.6254'

Satellite position 7
'  34431848.99857   5085624.04357  21266430.3444   21266433.2854'
'     -2157.3284'

Satellite position 8
'  10350212.86953   3216363.32353  24628572.8524   24628577.7974'
'     -289

In [34]:
# ============================================================
# STAGE 05 — STEP 7
# RECONSTRUCT ALL SATELLITES IN THE FIRST EPOCH
# ============================================================

observation_types = ["L1", "L2", "C1", "P2", "P1", "D1"]

# Ambil waktu dan daftar satelit dari epoch pertama
first_epoch = df_epochs.iloc[0]

epoch_time = first_epoch["Epoch_Time_GPS"]
satellite_ids = first_epoch["Satellite_IDs_Normalized"]

n_satellites = len(satellite_ids)
n_observations = len(observation_types)

first_observation_index = header_end_index + 2

reconstructed_data = []

for sat_index, satellite_id in enumerate(satellite_ids):

    start = first_observation_index + sat_index * 2

    first_line = raw_lines[start].rstrip("\r\n").ljust(80)
    continuation_line = raw_lines[start + 1].rstrip("\r\n").ljust(80)

    raw_fields = [
        first_line[i:i + 16]
        for i in range(0, 80, 16)
    ]

    raw_fields.append(continuation_line[0:16])

    for obs_type, field in zip(observation_types, raw_fields):

        value_text = field[:14].strip()

        try:
            value = float(value_text) if value_text else np.nan
        except ValueError:
            value = np.nan

        reconstructed_data.append({
            "Epoch_Time_GPS": epoch_time,
            "Satellite_ID": satellite_id,
            "Observation_Type": obs_type,
            "Value": value,
            "LLI": field[14],
            "SSI": field[15]
        })

df_first_epoch_reconstructed = pd.DataFrame(reconstructed_data)

print("FIRST EPOCH RECONSTRUCTION SUMMARY")
print("=" * 80)
print(f"Epoch time GPS       : {epoch_time}")
print(f"Number of satellites : {n_satellites}")
print(f"Observation types    : {n_observations}")
print(f"Expected rows        : {n_satellites * n_observations}")
print(f"Actual rows          : {len(df_first_epoch_reconstructed)}")
print(f"Unique satellites    : {df_first_epoch_reconstructed['Satellite_ID'].nunique()}")

display(df_first_epoch_reconstructed)

FIRST EPOCH RECONSTRUCTION SUMMARY
Epoch time GPS       : 2003-10-27 00:00:30
Number of satellites : 9
Observation types    : 6
Expected rows        : 54
Actual rows          : 54
Unique satellites    : 9


,Epoch_Time_GPS,Satellite_ID,Observation_Type,Value,LLI,SSI
0,2003-10-27 00:00:30,G21,L1,7.272414e+06,5,7
1,2003-10-27 00:00:30,G21,L2,5.719375e+06,5,7
2,2003-10-27 00:00:30,G21,C1,2.065996e+07,4,
3,2003-10-27 00:00:30,G21,P2,2.065996e+07,4,
4,2003-10-27 00:00:30,G21,P1,NaN,,
5,2003-10-27 00:00:30,G21,D1,-3.921094e+03,4,
6,2003-10-27 00:00:30,G06,L1,3.544996e+07,5,5
7,2003-10-27 00:00:30,G06,L2,2.632699e+06,5,5
8,2003-10-27 00:00:30,G06,C1,2.321546e+07,4,
9,2003-10-27 00:00:30,G06,P2,2.321547e+07,4,


In [35]:
# ============================================================
# STAGE 05 — STEP 8
# FIRST EPOCH RECONSTRUCTION VALIDATION
# ============================================================

df_check = df_first_epoch_reconstructed.copy()

# Pemeriksaan jumlah record per satelit
satellite_record_counts = (
    df_check.groupby("Satellite_ID")
    .size()
)

# Pemeriksaan jumlah record per tipe observasi
observation_record_counts = (
    df_check.groupby("Observation_Type")
    .size()
)

# Pemeriksaan kombinasi satelit dan tipe observasi
duplicate_combinations = df_check.duplicated(
    subset=["Satellite_ID", "Observation_Type"]
).sum()

# Pemeriksaan nilai kosong
missing_values = (
    df_check.groupby("Observation_Type")["Value"]
    .apply(lambda x: x.isna().sum())
)

print("FIRST EPOCH RECONSTRUCTION VALIDATION")
print("=" * 80)

print("\nRECORD COUNT PER SATELLITE")
print(satellite_record_counts.to_string())

print("\nRECORD COUNT PER OBSERVATION TYPE")
print(observation_record_counts.to_string())

print("\nMISSING VALUE COUNT")
print(missing_values.to_string())

print("\nVALIDATION SUMMARY")
print("-" * 80)
print(f"Total records              : {len(df_check)}")
print(f"Unique satellites          : {df_check['Satellite_ID'].nunique()}")
print(f"Unique observation types   : {df_check['Observation_Type'].nunique()}")
print(f"Duplicate combinations     : {duplicate_combinations}")
print(f"All satellites have 6 rows : {(satellite_record_counts == 6).all()}")
print(f"All types have 9 rows      : {(observation_record_counts == 9).all()}")
print(f"Only P1 is entirely missing: (check output)")

FIRST EPOCH RECONSTRUCTION VALIDATION

RECORD COUNT PER SATELLITE
Satellite_ID
G05    6
G06    6
G14    6
G15    6
G18    6
G21    6
G23    6
G25    6
G30    6

RECORD COUNT PER OBSERVATION TYPE
Observation_Type
C1    9
D1    9
L1    9
L2    9
P1    9
P2    9

MISSING VALUE COUNT
Observation_Type
C1    0
D1    0
L1    0
L2    0
P1    9
P2    0

VALIDATION SUMMARY
--------------------------------------------------------------------------------
Total records              : 54
Unique satellites          : 9
Unique observation types   : 6
Duplicate combinations     : 0
All satellites have 6 rows : True
All types have 9 rows      : True
Only P1 is entirely missing: (check output)


In [37]:
# ============================================================
# STAGE 05 — PREPARE RAW RINEX INPUT
# ============================================================

with open(MAIN_RINEX_FILE, "r") as f:
    lines = f.readlines()

print("RAW RINEX INPUT")
print("=" * 60)
print(f"File              : {MAIN_RINEX_FILE.name}")
print(f"Total physical lines : {len(lines):,}")
print(f"Header detected   : {any('END OF HEADER' in x for x in lines)}")

RAW RINEX INPUT
File              : bako3000.03o
Total physical lines : 51,404
Header detected   : True


In [38]:
df_parsed_doy300, df_parsed_epochs_doy300 = (
    parse_rinex_observations(lines, observation_types)
)

In [40]:
# ============================================================
# STAGE 05 — INSPECT PARSED DATAFRAME STRUCTURE
# ============================================================

print("DF_PARSED_DOY300")
print("=" * 60)

print("Column names:")
print(df_parsed_doy300.columns.tolist())

print("\nData types:")
print(df_parsed_doy300.dtypes)

print("\nFirst 12 records:")
display(df_parsed_doy300.head(12))

print("\nEPOCH DATAFRAME")
print("=" * 60)

print("Column names:")
print(df_parsed_epochs_doy300.columns.tolist())

display(df_parsed_epochs_doy300.head(5))

DF_PARSED_DOY300
Column names:
['Timestamp_GPS', 'Epoch_Flag', 'Satellite_ID', 'Observation_Type', 'Value', 'LLI', 'SSI']

Data types:
Timestamp_GPS       datetime64[us]
Epoch_Flag                   int64
Satellite_ID                   str
Observation_Type               str
Value                      float64
LLI                            str
SSI                            str
dtype: object

First 12 records:


,Timestamp_GPS,Epoch_Flag,Satellite_ID,Observation_Type,Value,LLI,SSI
0,2003-10-27 00:00:30,0,G21,L1,7.272414e+06,5,7
1,2003-10-27 00:00:30,0,G21,L2,5.719375e+06,5,7
2,2003-10-27 00:00:30,0,G21,C1,2.065996e+07,4,NaN
3,2003-10-27 00:00:30,0,G21,P2,2.065996e+07,4,NaN
4,2003-10-27 00:00:30,0,G21,P1,NaN,NaN,NaN
5,2003-10-27 00:00:30,0,G21,D1,-3.921094e+03,4,NaN
6,2003-10-27 00:00:30,0,G06,L1,3.544996e+07,5,5
7,2003-10-27 00:00:30,0,G06,L2,2.632699e+06,5,5
8,2003-10-27 00:00:30,0,G06,C1,2.321546e+07,4,NaN
9,2003-10-27 00:00:30,0,G06,P2,2.321547e+07,4,NaN



EPOCH DATAFRAME
Column names:
['Timestamp_GPS', 'Epoch_Flag', 'Declared_Satellites']


,Timestamp_GPS,Epoch_Flag,Declared_Satellites
0,2003-10-27 00:00:30.000,0,9
1,2003-10-27 00:01:00.000,0,9
2,2003-10-27 00:01:30.000,0,9
3,2003-10-27 00:02:00.001,0,9
4,2003-10-27 00:02:30.001,0,9


In [41]:
# ============================================================
# STAGE 05 — PARSED OBSERVATION SUMMARY
# ============================================================

print("OBSERVATION RECORD DISTRIBUTION")
print("=" * 60)

print("\nRecords per observation type:")
display(
    df_parsed_doy300["Observation_Type"]
    .value_counts()
    .sort_index()
    .rename("Record_Count")
    .to_frame()
)

print("\nMissing values per observation type:")
display(
    df_parsed_doy300
    .groupby("Observation_Type")["Value"]
    .apply(lambda x: x.isna().sum())
    .rename("Missing_Value_Count")
    .to_frame()
)

print("\nMissing LLI and SSI:")
display(
    df_parsed_doy300[
        ["LLI", "SSI"]
    ].isna().sum().to_frame("Missing_Count")
)

OBSERVATION RECORD DISTRIBUTION

Records per observation type:


,Record_Count
Observation_Type,
C1,24254
D1,24254
L1,24254
L2,24254
P1,24254
P2,24254



Missing values per observation type:


,Missing_Value_Count
Observation_Type,
C1,0
D1,0
L1,17
L2,374
P1,24254
P2,374



Missing LLI and SSI:


,Missing_Count
LLI,25019
SSI,97407


In [42]:
# ============================================================
# STAGE 05 — EXPORT PARSED OBSERVATIONS
# ============================================================

parsed_output = (
    OUTPUT_DIR
    / "parsed_observations"
)

parsed_output.mkdir(parents=True, exist_ok=True)

observation_file = (
    parsed_output
    / "parsed_observations_doy300.csv"
)

epoch_file = (
    parsed_output
    / "parsed_epochs_doy300.csv"
)

df_parsed_doy300.to_csv(
    observation_file,
    index=False
)

df_parsed_epochs_doy300.to_csv(
    epoch_file,
    index=False
)

print("PARSER OUTPUT EXPORT")
print("=" * 60)

print(f"Observation file : {observation_file}")
print(f"Epoch file       : {epoch_file}")

print(f"\nObservation records exported : {len(df_parsed_doy300):,}")
print(f"Epoch records exported       : {len(df_parsed_epochs_doy300):,}")

print(f"\nObservation file exists : {observation_file.exists()}")
print(f"Epoch file exists       : {epoch_file.exists()}")

PARSER OUTPUT EXPORT
Observation file : C:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final\Output\Notebook_01\parsed_observations\parsed_observations_doy300.csv
Epoch file       : C:\SKRIPSI AKHIR\ALL DATA\GNSS\CDDIS_GNSS Data Daily\HALLOWEEN STORM 2003\Pipeline_Final\Output\Notebook_01\parsed_observations\parsed_epochs_doy300.csv

Observation records exported : 145,524
Epoch records exported       : 2,878

Observation file exists : True
Epoch file exists       : True


In [43]:
# ============================================================
# STAGE 05 — EXPORT READ-BACK VERIFICATION
# ============================================================

df_obs_check = pd.read_csv(observation_file)
df_epoch_check = pd.read_csv(epoch_file)

print("EXPORT READ-BACK VERIFICATION")
print("=" * 60)

print(f"Observation records : {len(df_obs_check):,}")
print(f"Epoch records       : {len(df_epoch_check):,}")

print(
    "Observation count preserved:",
    len(df_obs_check) == len(df_parsed_doy300)
)

print(
    "Epoch count preserved:",
    len(df_epoch_check) == len(df_parsed_epochs_doy300)
)

EXPORT READ-BACK VERIFICATION
Observation records : 145,524
Epoch records       : 2,878
Observation count preserved: True
Epoch count preserved: True


# README — Pipeline 01: RINEX Data Ingestion and Parser Development

## 1. Overview

Pipeline 01 merupakan tahap awal pengolahan data GNSS RINEX dalam penelitian *Multi Parameter Observations of Solar Storm Impacts on the Indonesian Equator Ionosphere*.

Pipeline ini bertujuan untuk mempelajari struktur berkas RINEX, mengekstraksi metadata header, mengidentifikasi struktur epoch dan rekaman observasi satelit, serta mengembangkan parser manual untuk menghasilkan tabel observasi GNSS yang terstruktur.

Pengembangan dan pengujian awal dilakukan menggunakan data stasiun BAKO pada 27 Oktober 2003 (DOY 300), sebagai sampel pengembangan parser.

**Notebook utama:**

`01_RINEX_Data_Ingestion_and_Parser_Development.ipynb`

## 2. Objectives

Tujuan Pipeline 01 adalah:

1. Menetapkan direktori proyek dan memilih berkas RINEX yang digunakan.
2. Mengekstraksi metadata dari header RINEX.
3. Menginterpretasikan tipe observasi berdasarkan definisi header.
4. Menganalisis struktur epoch, epoch flag, dan daftar satelit.
5. Mempelajari format rekaman observasi satelit.
6. Mengembangkan mekanisme pembacaan rekaman observasi yang mencakup continuation lines.
7. Mengembangkan parser manual untuk menghasilkan tabel observasi terstruktur.
8. Melakukan pemeriksaan awal terhadap hasil parsing.
9. Mengekspor hasil parsing dan metadata untuk tahap validasi berikutnya.

## 3. Input Dataset

| Parameter | Keterangan |
|---|---|
| Station | BAKO |
| Data format | RINEX 2.10 |
| Satellite system | GPS |
| Observation date | 27 October 2003 |
| Day of Year | DOY 300 |
| Sampling interval | Nominally 30 seconds |
| Observation types | L1, L2, C1, P2, P1, D1 |
| Input file | `bako3000.03o` |

Data mentah berada pada direktori:

`data/raw/DOY300/bako3000.03o`

Seluruh proses pengembangan parser pada tahap ini menggunakan berkas mentah sebagai input tanpa mengubah sumber data asli.

## 4. Processing Workflow

Pipeline 01 disusun berdasarkan sepuluh tahapan berikut.

### 4.1 Notebook Initialization and Dataset Selection

Menetapkan direktori proyek, lokasi data mentah, lokasi keluaran, dan berkas RINEX yang digunakan sebagai sampel pengembangan.

### 4.2 RINEX Header Extraction

Membaca bagian header hingga `END OF HEADER` dan mengekstraksi metadata utama, termasuk:

- RINEX version and file type
- Satellite system
- Marker name and number
- Observer and agency
- Receiver information
- Antenna information
- Approximate station coordinates
- Antenna reference point
- Observation types
- Time of first observation

### 4.3 Header Interpretation and Observation Schema

Menginterpretasikan urutan tipe observasi berdasarkan informasi `# / TYPES OF OBSERV` pada header.

Urutan observasi yang digunakan:

`L1, L2, C1, P2, P1, D1`

Setiap nilai observasi direpresentasikan menggunakan struktur fixed-width RINEX yang terdiri atas nilai numerik, Loss of Lock Indicator (LLI), dan Signal Strength Indicator (SSI).

### 4.4 Epoch Record Structure Analysis

Menganalisis struktur epoch, meliputi:

- Epoch timestamp
- Epoch flag
- Number of declared satellites
- Satellite identification list
- Temporal continuity
- Duplicate timestamps
- Observation interval

### 4.5 Satellite Observation Record Analysis

Mempelajari hubungan antara satelit dan tipe observasi melalui pembacaan fixed-width observation fields.

Setiap satellite instance direpresentasikan menggunakan enam tipe observasi sesuai skema header.

### 4.6 Continuation Record Handling

Mengembangkan mekanisme pembacaan baris lanjutan pada rekaman observasi.

Pembacaan observation records mempertimbangkan jumlah tipe observasi yang didefinisikan pada header dan kapasitas lima observation fields per physical line.

Penanganan continuation pada daftar satelit perlu dibedakan dari continuation pada observation records.

### 4.7 Manual RINEX Parser Development

Mengintegrasikan pembacaan epoch, identifikasi satelit, dan ekstraksi observation fields ke dalam fungsi parser manual.

Hasil parsing disusun dalam bentuk tabel terstruktur dengan kolom:

- `Timestamp_GPS`
- `Epoch_Flag`
- `Satellite_ID`
- `Observation_Type`
- `Value`
- `LLI`
- `SSI`

### 4.8 Parser Execution and Exception Handling

Menjalankan parser pada berkas RINEX BAKO DOY300 dan menangani kondisi pembacaan yang tidak sesuai dengan asumsi parser.

Implementasi awal ini berfokus pada epoch flag 0 dan struktur data sampel yang digunakan.

### 4.9 Preliminary Parser Diagnostics

Melakukan pemeriksaan awal terhadap:

- Jumlah epoch hasil parsing
- Jumlah observation records
- Distribusi observation types
- Missing observation values
- Missing LLI and SSI values
- Konsistensi jumlah record sebelum dan sesudah ekspor

Pemeriksaan ini merupakan diagnostik internal awal, bukan validasi independen parser.

### 4.10 Export and Documentation

Menyimpan metadata, struktur epoch, dan hasil parsing ke dalam direktori keluaran Pipeline 01.

## 5. Preliminary Parsing Results

Hasil eksekusi parser manual untuk BAKO DOY300:

| Parameter | Result |
|---|---:|
| Total physical lines | 51,404 |
| Header lines | 18 |
| Epoch records | 2,878 |
| Satellite instances | 24,254 |
| Observation types | 6 |
| Total observation records | 145,524 |

Setiap satellite instance menghasilkan enam baris observasi, sehingga:

24,254 × 6 = 145,524 observation records.

### 5.1 Observation Value Availability

| Observation Type | Total Records | Missing Values |
|---|---:|---:|
| L1 | 24,254 | 17 |
| L2 | 24,254 | 374 |
| C1 | 24,254 | 0 |
| P2 | 24,254 | 374 |
| P1 | 24,254 | 24,254 |
| D1 | 24,254 | 0 |

P1 tercatat kosong pada seluruh satellite instances dalam berkas sampel ini. Kondisi tersebut dipertahankan sebagai missing values dan tidak diganti dengan nol.

### 5.2 Time Reference

Timestamp hasil parsing menggunakan basis waktu GPS (`Timestamp_GPS`).

Tidak dilakukan konversi otomatis dari GPS Time ke UTC pada Pipeline 01.

## 6. Output Directory

Hasil Pipeline 01 disimpan pada:

`Output/Notebook_01/`

Struktur direktori keluaran:

```text
Output/
└── Notebook_01/
    ├── header_metadata/
    │   ├── header_metadata_doy300.csv
    │   ├── header_metadata_doy300.json
    │   └── header_metadata_validation_doy300.csv
    │
    ├── epoch_structure/
    │   └── epoch_structure_qc_doy300.csv
    │
    ├── parsed_observations/
    │   ├── parsed_observations_doy300.csv
    │   └── parsed_epochs_doy300.csv
    │
    ├── diagnostics/
    └── parsing_logs/
```

### Main Output Files

| File | Description |
|---|---|
| `header_metadata_doy300.csv` | Structured RINEX header metadata |
| `header_metadata_doy300.json` | Header metadata in JSON format |
| `header_metadata_validation_doy300.csv` | Internal header metadata checks |
| `epoch_structure_qc_doy300.csv` | Preliminary epoch structure information |
| `parsed_observations_doy300.csv` | Structured satellite observation records |
| `parsed_epochs_doy300.csv` | Parsed epoch information |

## 7. Limitations and Validation Status

Pipeline 01 menghasilkan parser manual dan keluaran terstruktur untuk sampel BAKO DOY300.

Beberapa batasan yang perlu diperhatikan:

1. Pengembangan parser dilakukan menggunakan satu berkas sampel, sehingga belum membuktikan kompatibilitas dengan seluruh berkas RINEX dalam dataset penelitian.
2. Penanganan observation continuation lines telah diterapkan untuk skema enam tipe observasi.
3. Penanganan satellite-list continuation belum diuji secara independen pada berkas yang memerlukannya.
4. Parser pada tahap ini difokuskan pada epoch flag 0.
5. Pemeriksaan jumlah record dan read-back ekspor hanya menunjukkan konsistensi internal.
6. Kesesuaian nilai hasil parsing dengan seluruh rekaman pada RINEX mentah belum ditetapkan melalui validasi independen.

Dengan demikian, hasil Pipeline 01 berstatus sebagai **preliminary manual parsing output**.

## 8. Next Pipeline

Hasil Pipeline 01 digunakan sebagai masukan untuk:

**Pipeline 02 — RINEX Parser Validation and Data Reconciliation**

Tahap berikutnya berfokus pada validasi parser manual dengan membandingkan hasil parsing terhadap struktur dan nilai pada berkas RINEX mentah serta hasil pembacaan menggunakan GeoRinex.

Validasi mencakup timestamp, satellite identification, observation values, LLI, SSI, dan rekonsiliasi jumlah rekaman.

## 9. Reproducibility

Untuk mereproduksi hasil Pipeline 01:

1. Pastikan berkas `bako3000.03o` tersedia pada direktori `data/raw/DOY300/`.
2. Pastikan konfigurasi direktori proyek sesuai dengan lingkungan kerja.
3. Buka notebook `01_RINEX_Data_Ingestion_and_Parser_Development.ipynb`.
4. Jalankan tahapan notebook secara berurutan.
5. Periksa hasil parsing dan diagnostik awal.
6. Pastikan file keluaran tersimpan pada `Output/Notebook_01/`.

**Pipeline status:** Completed for initial parser development on BAKO DOY300. Independent parser validation is assigned to Pipeline 02.